# GenAI Platform Lab: validação Databricks
Notebook gerado do job canônico e fixture sintética. Não requer AWS, segredos ou upload adicional. Execute em Free Edition.
A última célula cria/reutiliza o Volume `genai_platform_lab` no catálogo/schema atual e sobrescreve somente sua pasta `fixture_daily`.

In [ ]:
"""Shared UTC/decimal transformation; works with local Spark and Databricks."""
import argparse

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ArrayType, DoubleType, IntegerType, LongType, StringType, StructField, StructType,
)

EVENT_SCHEMA = StructType([
    StructField("record_id", StringType()), StructField("timestamp", StringType()),
    StructField("application", StringType()), StructField("model", StringType()),
    StructField("status", StringType()), StructField("prompt_tokens", LongType()),
    StructField("completion_tokens", LongType()), StructField("total_tokens", LongType()),
    StructField("duration_ms", DoubleType()), StructField("spend", StringType()),
])
EXPORT_SCHEMA = StructType([
    StructField("schema_version", IntegerType()),
    StructField("events", ArrayType(EVENT_SCHEMA)),
])

def read_export(spark, path):
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    document = spark.read.schema(EXPORT_SCHEMA).option("multiline", True).json(path)
    if document.count() != 1 or document.filter(
        F.col("schema_version").isNull() | (F.col("schema_version") != 1) |
        F.col("events").isNull()
    ).count():
        raise ValueError("Expected one schema_version=1 export")
    return document.select(F.explode("events").alias("event")).select("event.*")

def aggregate_usage(frame):
    frame.sparkSession.conf.set("spark.sql.session.timeZone", "UTC")
    # Identical duplicates are harmless; conflicting records must stop the batch.
    frame = frame.dropDuplicates()
    if frame.groupBy("record_id").count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Conflicting duplicate record_id")
    invalid = (F.col("record_id").isNull() | F.col("timestamp").isNull() |
               F.col("application").isNull() | F.col("model").isNull() |
               ~F.col("timestamp").rlike(r"(Z|\+00:00)$") |
               (F.col("status").isNotNull() & ~F.col("status").isin("success", "failure")))
    for name in ("prompt_tokens", "completion_tokens", "total_tokens", "duration_ms"):
        invalid = invalid | (F.col(name) < 0)
    if frame.filter(invalid).limit(1).count():
        raise ValueError("Invalid or non-UTC event")
    frame = frame.withColumn("day", F.to_date(F.to_timestamp("timestamp")))
    frame = frame.withColumn("money", F.col("spend").cast("decimal(24,12)"))
    if frame.filter(F.col("day").isNull() |
        (F.col("spend").isNotNull() & F.col("money").isNull()) |
        (F.col("money") < 0)).limit(1).count():
        raise ValueError("Invalid timestamp or decimal")
    expressions = [
        F.count("*").alias("requests"), F.count("status").alias("status_known"),
        F.sum(F.when(F.col("status") == "failure", 1)
              .when(F.col("status") == "success", 0)).alias("errors"),
        F.sum("money").alias("spend"), F.count("money").alias("spend_known"),
        F.avg("duration_ms").alias("latency_mean_ms"),
    ]
    for name in ("prompt_tokens", "completion_tokens", "total_tokens", "duration_ms"):
        expressions.extend([F.sum(name).alias(name), F.count(name).alias(name + "_known")])
    return frame.groupBy("day", "application", "model").agg(*expressions)


In [ ]:
fixture = {'schema_version': 1, 'coverage': {'state': 'synthetic', 'source': 'hand-calculated fixture'}, 'events': [{'record_id': 'synthetic-1', 'timestamp': '2026-01-01T12:00:00+00:00', 'application': 'app-a', 'model': 'primary', 'status': 'success', 'prompt_tokens': 8, 'completion_tokens': 4, 'total_tokens': 12, 'duration_ms': 1000, 'spend': '0.1'}, {'record_id': 'synthetic-2', 'timestamp': '2026-01-01T14:00:00+00:00', 'application': 'app-a', 'model': 'primary', 'status': None, 'prompt_tokens': 12, 'completion_tokens': 6, 'total_tokens': 18, 'duration_ms': 2000, 'spend': '0.2'}, {'record_id': 'synthetic-3', 'timestamp': '2026-01-01T12:00:00+00:00', 'application': 'app-b', 'model': 'reserve', 'status': 'failure', 'prompt_tokens': None, 'completion_tokens': None, 'total_tokens': None, 'duration_ms': None, 'spend': None}, {'record_id': 'synthetic-1', 'timestamp': '2026-01-01T12:00:00+00:00', 'application': 'app-a', 'model': 'primary', 'status': 'success', 'prompt_tokens': 8, 'completion_tokens': 4, 'total_tokens': 12, 'duration_ms': 1000, 'spend': '0.1'}]}
for event in fixture['events']:
    if event['duration_ms'] is not None:
        event['duration_ms'] = float(event['duration_ms'])
frame = spark.createDataFrame(fixture['events'], EVENT_SCHEMA)
result = aggregate_usage(frame)
display(result.orderBy('day', 'application', 'model'))

In [ ]:
rows = {(str(r.day), r.application, r.model): r.asDict() for r in result.collect()}
a = rows[('2026-01-01', 'app-a', 'primary')]
b = rows[('2026-01-01', 'app-b', 'reserve')]
assert a['requests'] == 2 and a['total_tokens'] == 30
assert str(a['spend']) == '0.300000000000'
assert a['status_known'] == 1 and a['errors'] == 0
assert a['latency_mean_ms'] == 1500
assert b['spend'] is None and b['spend_known'] == 0 and b['errors'] == 1
print('DATABRICKS_FIXTURE_OK: resultados iguais ao PySpark local')

In [ ]:
namespace = spark.sql('SELECT current_catalog() AS catalog, current_schema() AS schema').first()
spark.sql('CREATE VOLUME IF NOT EXISTS genai_platform_lab')
output = (f'/Volumes/{namespace.catalog}/{namespace.schema}/'
          'genai_platform_lab/fixture_daily')
result.write.mode('overwrite').parquet(output)
first = sorted(str(r) for r in spark.read.parquet(output).collect())
result.write.mode('overwrite').parquet(output)
assert first == sorted(str(r) for r in spark.read.parquet(output).collect())
print('DATABRICKS_PARQUET_OK: reexecução idempotente')